# Clase de ETL: de datos en bruto a datos útiles

**Extract · Transform · Load** (extraer · transformar · cargar) con datos reales de Eurostat sobre el gasto público en educación (% del PIB).

No necesitas saber Python. Vas a **leer** código, **ejecutarlo** y **probar** pequeños cambios. Todo se explica sobre la marcha.

| Parte | Qué | Tiempo |
|---|---|---|
| 0 | Preparación y tu predicción | 5 min |
| 1 | DataFrames en diez minutos | 15 min |
| 2 | **Extract**: obtener los datos y mirarlos | 20 min |
| 3 | **Transform**: limpiarlos, decidir, hacer preguntas, encontrar los errores | 40 min |
| 4 | **Load**: guardarlos y hacer un gráfico | 20 min |
| 5 | Reflexión | 10 min |
| 6 | *Opcional:* práctica con películas y valoraciones (MovieLens) | 45 min, para casa |

### Cómo usar este notebook

- Ejecuta una celda con **Shift + Enter**. Ejecuta las celdas **en orden, desde arriba**.
- Las celdas marcadas como **Tu turno** tienen un hueco que debes rellenar: sustituye `None` por tu propio código, ejecuta la celda y después ejecuta la celda de **comprobación** que hay justo debajo. Te dirá `[OK]` o te dará una pista.
- Si te sale un error en rojo, lee la **última línea**: normalmente te dice qué está mal. Pregunta a tu compañero/a o a tu profesor/a.
- No puedes romper nada: si el notebook se lía, usa *Kernel → Restart Kernel and Run All Cells* (los ejercicios simplemente vuelven a mostrar las pistas).

## Parte 0 · Preparación y tu predicción

**Antes de ejecutar nada**, escribe en un papel:

1. ¿Cuántos valores crees que faltan en los datos?
2. ¿Qué país europeo gasta más en educación?
3. ¿Cómo crees que es el archivo original?

Comprobaremos tus predicciones al final.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Encuentra la carpeta del proyecto (funciona tanto si Jupyter se inició en notebooks/ como en la raíz del proyecto)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # nos permite usar el paquete auxiliar `etl`

RAW = ROOT / "data" / "raw"
REF = ROOT / "data" / "reference"
EXERCISES = ROOT / "data" / "exercises"
DOWNLOADS = ROOT / "data" / "downloads"
PROCESSED = ROOT / "data" / "processed"
OUTPUT = ROOT / "output"

pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def check(name, ok, hint=""):
    """Te dice si tu respuesta es correcta."""
    if ok:
        print(f"[OK] {name}")
    else:
        print(f"[todavía no] {name}. Pista: {hint}")


print("Preparación hecha. Carpeta del proyecto:", ROOT)

## Parte 1 · DataFrames en diez minutos

ETL trabaja con **tablas**. En Python (con la librería **pandas**) una tabla se llama **DataFrame**:

- las **filas** son observaciones (aquí: un país en un año),
- las **columnas** son variables (país, año, gasto),
- **cada columna tiene un solo tipo** (texto, números enteros, decimales…),
- las filas y las columnas tienen **etiquetas**.

Piensa en ella como una hoja de cálculo estricta, o como una matriz con nombres. Una sola columna es una **Series** (un vector con etiquetas).

Vamos a construir una pequeñita a mano.

In [ ]:
toy = pd.DataFrame({
    "country": ["Denmark", "Spain", "Finland"],
    "year": [2021, 2021, 2021],
    "spending": [6.44, 4.69, 6.16],      # gasto público en educación, % del PIB
})
toy

Todo DataFrame conoce su **forma** (*shape*: filas, columnas) y el **tipo** de cada columna:

In [ ]:
print("forma:", toy.shape)
print(toy.dtypes)

Tres operaciones cubren casi todo lo que haremos hoy: **seleccionar una columna**, **filtrar filas** y **resumir**.

In [ ]:
toy["spending"]                          # selecciona una columna (una Series)

In [ ]:
toy[toy["spending"] > 6]                 # filtra: se queda con las filas donde la condición es True

In [ ]:
toy["spending"].mean()                   # resume

### Valores ausentes

Los datos reales tienen huecos. pandas escribe un número ausente como **`NaN`** ("not a number", no es un número). Un `NaN` significa *no medido*, **no cero**. Dos cosas útiles:

- Para detectarlo, usa `isnull()` (nunca `==`, porque `NaN` nunca es igual a nada, ni siquiera a sí mismo).
- La mayoría de los resúmenes de pandas **se saltan** los valores ausentes.

In [ ]:
with_gap = toy.copy()
with_gap.loc[1, "spending"] = np.nan     # hacemos como si el valor de España no estuviera disponible

print(with_gap["spending"].isnull())
print("media ignorando el hueco:", with_gap["spending"].mean())

### Tu turno 1.1

A partir de `toy`, quédate solo con las filas donde `spending` es **menor que 6.2**. Guarda el resultado en `answer_1_1`.

<details><summary>Pista</summary>

Mira el filtro de arriba (`toy[toy["spending"] > 6]`) y cambia la condición.
</details>

In [ ]:
# Sustituye None por tu código
answer_1_1 = None

In [ ]:
check("Ejercicio 1.1",
      answer_1_1 is not None and list(answer_1_1["country"]) == ["Spain", "Finland"],
      'filtra con una condición como toy["spending"] < ...')

### Tu turno 1.2

¿Cuál es el gasto medio en `toy`, **redondeado a 2 decimales**? Guárdalo en `answer_1_2`.

<details><summary>Pista</summary>

`round(number, 2)` redondea un número; la media es `toy["spending"].mean()`.
</details>

In [ ]:
answer_1_2 = None

In [ ]:
check("Ejercicio 1.2", answer_1_2 is not None and answer_1_2 == 5.76,'calcula la media de la columna y después round(..., 2)')

> **¿Qué librería?** Hay varias librerías para DataFrames. Usamos **pandas** (la más extendida). **polars** es una alternativa más nueva y más rápida; si acabas antes, el Apéndice A muestra las mismas operaciones en polars.

## Parte 2 · Extract (extraer)

**Extract** significa llevar los datos desde su fuente hasta nuestro ordenador **sin modificarlos**, y después mirarlos con atención.

Nuestros datos vienen de **Eurostat** (la oficina estadística de la UE), conjunto de datos [`educ_uoe_fine06`](https://ec.europa.eu/eurostat/databrowser/view/educ_uoe_fine06/default/table): *gasto público en educación como % del PIB*, todos los niveles desde educación infantil hasta educación superior, 2012–2023. Son datos abiertos. Tenemos dos archivos en `data/raw/`, descargados de la API de Eurostat:

- `educ_uoe_fine06_long.csv`: formato **largo** (SDMX-CSV), una fila por país y año.
- `educ_uoe_fine06.tsv`: el formato **en bruto** tal como lo sirve Eurostat: ancho, con códigos empaquetados juntos y notas pegadas a los números.

Primero, vamos a mirar los archivos **como texto plano**, antes de que pandas interprete nada:

In [ ]:
for name in ["educ_uoe_fine06_long.csv", "educ_uoe_fine06.tsv"]:
    print(name)
    for line in (RAW / name).read_text(encoding="utf-8").splitlines()[:3]:
        print("   ", repr(line[:110]))       # repr muestra caracteres ocultos como \t (un tabulador)
    print()

Fíjate en las diferencias: el CSV separa los campos con **comas** y pone un valor por línea (con muchas columnas, algunas repetidas como código y como etiqueta); el TSV separa los campos con **tabuladores** (`\t`), tiene una **columna por año**, empaqueta cuatro códigos en su primera columna (`A,PC_GDP,ED02-8,AT`), y algunas celdas son como `5.16 d` o `: d`.

### 2.1 Leer el CSV largo

El CSV tiene 19 columnas, pero solo necesitamos cuatro: el código del país (`geo`), el año (`TIME_PERIOD`), el valor (`OBS_VALUE`) y su flag (`OBS_FLAG`). `usecols` se queda solo con esas.

In [ ]:
edu = pd.read_csv(RAW / "educ_uoe_fine06_long.csv", usecols=["geo", "TIME_PERIOD", "OBS_VALUE", "OBS_FLAG"])
edu.head()

In [ ]:
print("forma:", edu.shape)
print(edu.dtypes)
edu.describe()

### Tu turno 2.1

¿Cuántos valores **faltan** en la columna `OBS_VALUE` de `edu`? Guarda el número en `answer_2_1`.

<details><summary>Pista</summary>

Usa `isnull()` sobre la columna y después `sum()` (cada `True` cuenta como 1).
</details>

In [ ]:
answer_2_1 = None

In [ ]:
check("Ejercicio 2.1", answer_2_1 is not None and answer_2_1 == 40, 'isnull() sobre la columna y después sum()')

**Mira con más atención.** 37 países y agregados × 12 años deberían dar 444 filas, pero `edu` tiene menos. El CSV **omite** un país-año cuando no hay valor *ni* flag: esos valores ausentes no son celdas vacías, son **filas que faltan**. Contar los `NaN` no basta para saber cuántos datos faltan; también tienes que contar lo que *debería* estar.

In [ ]:
expected = edu["geo"].nunique() * edu["TIME_PERIOD"].nunique()
print("filas esperadas:", expected, "| filas en el archivo:", len(edu), "| filas que faltan:", expected - len(edu))

### 2.2 Leer el TSV en bruto

¿Qué pasa si leemos el TSV **como si fuera un CSV** (separador coma)?

In [ ]:
wrong = pd.read_csv(RAW / "educ_uoe_fine06.tsv")
print("forma:", wrong.shape)
wrong.head(3)

Un desastre: pandas buscó comas, y las únicas comas están **dentro de la primera columna** (`A,PC_GDP,ED02-8,AT`). Creó 4 columnas, y la última tiene el código del país y los doce años pegados con tabuladores. Tenemos que decirle que el separador es un **tabulador** (`sep="\t"`). Además lo leemos todo **como texto** (`dtype=str`) para que pandas no intente adivinar los tipos a partir de valores como `5.16 d`.

In [ ]:
raw = pd.read_csv(RAW / "educ_uoe_fine06.tsv", sep="\t", dtype=str)
raw.columns = [c.strip() for c in raw.columns]          # los nombres de las columnas tienen espacios sobrantes
print("forma:", raw.shape)
raw.iloc[:6, [0, 1, 2, 3, 4]]                           # unas cuantas filas y columnas

### Tu turno 2.2

¿Cuántos **años** (columnas, sin contar la primera) tiene `raw`? Guárdalo en `answer_2_2`.

<details><summary>Pista</summary>

`raw.shape` es `(filas, columnas)`. La primera columna contiene los códigos, no un año.
</details>

In [ ]:
answer_2_2 = None

In [ ]:
check("Ejercicio 2.2", answer_2_2 is not None and answer_2_2 == 12, "raw.shape es (filas, columnas); la primera columna no es un año")

### 2.3 Desempaquetar la primera columna

La primera columna se llama `freq,unit,isced11,geo\TIME_PERIOD`: cuatro códigos separados por comas, y un recordatorio de que las demás columnas son años (`TIME_PERIOD`). Por ejemplo, `A,PC_GDP,ED02-8,AT` significa:

| Código | Dimensión | Significado |
|---|---|---|
| `A` | `freq` | datos anuales |
| `PC_GDP` | `unit` | porcentaje del PIB |
| `ED02-8` | `isced11` | nivel educativo: de infantil a superior (todos los niveles) |
| `AT` | `geo` | Austria |

La partimos por las comas (`str.split(",", expand=True)` da una columna por cada trozo):

In [ ]:
key = raw.columns[0]
raw[["freq", "unit", "isced11", "geo"]] = raw[key].str.split(",", expand=True)
raw[["freq", "unit", "isced11", "geo"]].nunique()       # ¿cuántos valores distintos hay en cada columna?

`freq`, `unit` e `isced11` tienen **un único valor**: solo descargamos esa porción del conjunto de datos, así que no aportan información. Las eliminamos (junto con la columna empaquetada) y dejamos `geo` en primer lugar.

> Si hubieras descargado el conjunto de datos **entero**, `isced11` tendría más de 20 niveles educativos y primero tendrías que **filtrar** `isced11 == "ED02-8"`. Compruébalo siempre antes de eliminar una columna.

In [ ]:
raw = raw.drop(columns=[key, "freq", "unit", "isced11"])
raw = raw[["geo"] + [c for c in raw.columns if c != "geo"]]   # pone geo en primer lugar
print("forma:", raw.shape)
raw.iloc[:6, [0, 7, 8, 9, 10]]

### 2.4 (Opcional) Descárgalo tú

El notebook también puede descargar el conjunto de datos de la API de Eurostat. Por defecto pide solo nuestra porción (`A.PC_GDP.ED02-8.`: anual, % del PIB, todos los niveles, todos los países). Si no hay red o el conjunto de datos ha cambiado, simplemente te lo dice, y seguimos usando la copia de `data/raw/`.

In [ ]:
from etl.extract import download_eurostat

try:
    path = download_eurostat("educ_uoe_fine06", DOWNLOADS)
    print("Descargado en", path)
except RuntimeError as err:
    print(err)
    print("No pasa nada: seguimos trabajando con la copia de data/raw/.")

### Punto de control: Extract

Antes de seguir, ¿sabrías explicarle a tu compañero/a…

- ¿Qué significa `:` en estos datos? ¿Y las letras que van después de un número? (Pista: abre `data/reference/eurostat_flags.csv`.)
- ¿Qué es un **flag** (la `d`, `p`, `e`, `b` que va después de un número) y por qué no deberíamos simplemente tirarlo?
- ¿Por qué el CSV tiene menos filas que celdas tiene el TSV?
- ¿Qué problemas ves ya que tendremos que arreglar? Haz una lista.

## Parte 3 · Transform (transformar)

Aquí está el trabajo y **las decisiones**. Nuestra tabla en bruto `raw` es *ancha* (una columna por año) y sus celdas mezclan números y notas. La convertiremos en una **tabla larga y limpia** con una fila por país y año.

| Paso | Qué |
|---|---|
| 3.1 | Ancha → larga |
| 3.2 | Separar valor y flag |
| 3.3 | Corregir los tipos |
| 3.4 | Añadir los nombres de los países (una *unión*, *join*) |
| 3.5 | Quitar los agregados (EU27_2020…) |
| 3.6 | Seleccionar el periodo |
| 3.7 | **Decidir** qué hacer con los valores ausentes |
| 3.8 | Resumir, pivotar, ordenar |
| 3.9 | Comprobar el resultado |
| 3.10 | Encontrar los errores sembrados |

### 3.1 Ancha → larga (*melt*)

En lugar de una columna por año, queremos una columna `year` y una columna `raw_cell`. Esto se llama **melt** (o *unpivot*): "fundir" las columnas de años en filas.

In [ ]:
long = raw.melt(id_vars="geo", var_name="year", value_name="raw_cell")
print("forma:", long.shape)            # 37 códigos x 12 años = 444 filas
long.head()

### 3.2 Separar valor y flag

Una celda como `6.71 p` contiene el **número** y la **flag** (nota de calidad). Python puede partir un texto por los espacios:

In [ ]:
"6.71 p".split(" ", 1)

Hacemos eso con toda la columna (`str.split`). Después:

- una celda que empieza por `:` **no tiene valor** → la convertimos en un valor ausente;
- el resto lo convertimos en **número** (`pd.to_numeric`).

In [ ]:
parts = long["raw_cell"].str.strip().str.split(r"\s+", n=1, expand=True)
parts.head(8)                           # columna 0 = el número (o ':'), columna 1 = la flag (o vacía)

In [ ]:
long["value"] = pd.to_numeric(parts[0].where(parts[0] != ":"), errors="coerce")
long["flag"] = parts[1]
long.head(8)

### 3.3 Corregir los tipos

`year` todavía contiene **texto** (era un nombre de columna). Queremos números enteros.

### Tu turno 3.1

Convierte `long["year"]` a enteros (`astype(int)`). Guarda la columna convertida en `answer_3_1`.

<details><summary>Pista</summary>

`long["year"].astype(int)`
</details>

In [ ]:
answer_3_1 = None

In [ ]:
check("Ejercicio 3.1", answer_3_1 is not None and answer_3_1.dtype == "int64", 'usa astype(...) sobre la columna')
long["year"] = long["year"].astype(int)       # lo aplicamos para todo el mundo, para que los pasos siguientes funcionen
long.dtypes

### 3.4 Añadir los nombres de los países (unión: *join* / *merge*)

`ES`, `DK`… son códigos. Una pequeña **tabla de consulta** en `data/reference/geo_codes.csv` da el nombre de cada código e indica si es un agregado. **Unimos** las dos tablas por la columna que comparten, `geo`. (`how="left"` conserva todas nuestras filas, incluso las que no encuentran pareja.)

In [ ]:
codes = pd.read_csv(REF / "geo_codes.csv")
codes.head()

In [ ]:
tidy = long.merge(codes, on="geo", how="left")
tidy.head()

### Tu turno 3.2

¿Cuántas filas de `tidy` se han quedado **sin nombre de país** después de la unión? Guarda el número en `answer_3_2`. (Si la unión ha funcionado, debería ser 0.)

<details><summary>Pista</summary>

`isnull().sum()` sobre la columna `country`.
</details>

In [ ]:
answer_3_2 = None

In [ ]:
check("Ejercicio 3.2", answer_3_2 is not None and answer_3_2 == 0, 'isnull() sobre la columna country, después sum()')

> Recuerda esta comprobación. Una unión es un lugar clásico para los **errores silenciosos**: un código sin pareja simplemente deja una celda vacía.

### 3.5 Quitar los agregados

`EU27_2020` (la Unión Europea desde 2020) y `EU28` (la UE antes del Brexit) son **grupos de países**, no países. Si los dejamos, aparecerán en una clasificación de "países". Para nuestras preguntas nos quedamos solo con países de verdad:

In [ ]:
print(tidy.drop_duplicates("geo")["is_aggregate"].value_counts())
countries = tidy[~tidy["is_aggregate"]]
print("filas conservadas:", len(countries), "de", len(tidy))

### 3.6 Seleccionar el periodo

Los datos cubren 2012–2023. Nos quedamos con **los últimos diez años**, 2014–2023. (Los dos primeros años son los que tienen más huecos.) Elegir el periodo también es una decisión: apúntala.

In [ ]:
period = countries[countries["year"].between(2014, 2023)].copy()
print("forma:", period.shape)
period.head()

### 3.7 Valores ausentes: **una decisión**

¿Qué países tienen huecos, y cuántos?

In [ ]:
missing_by_country = period["value"].isnull().groupby(period["country"]).sum()
missing_by_country[missing_by_country > 0].sort_values(ascending=False)

¿Qué hacemos con ellos? Hay tres opciones. Veamos qué hace cada una con el **gasto medio** de algunos países:

In [ ]:
leave_missing = period.groupby("country")["value"].mean()                        # opción A: dejarlos como ausentes
fill_zero = period.assign(value=period["value"].fillna(0)).groupby("country")["value"].mean()   # opción B: rellenar con 0

pd.DataFrame({"A: dejar como ausente": leave_missing, "B: rellenar con 0": fill_zero}).loc[
    ["Türkiye", "United Kingdom", "Greece", "Denmark"]
]

Fíjate en **Türkiye** (Turquía): en esta tabla *no tiene ningún dato*. Rellenar con 0 dice que Turquía no gasta **nada** en educación. Fíjate en el **United Kingdom** (Reino Unido): dejó de enviar datos después de 2019 (salió de la UE en 2020), y con ceros su media baja un **40 %**. Grecia y Dinamarca también bajan.

| Opción | Qué hace | Riesgo |
|---|---|---|
| **A. Dejarlos como ausentes** | Las medias usan solo los datos que tenemos | Hay que recordar que algunas medias se basan en menos años |
| **B. Rellenar con 0** | Se inventa un valor | Falso: no medido no es cero. Hace bajar las medias |
| **C. Eliminar las filas** | Mismas medias que A, pero se pierde la estructura país-año | Cuesta más ver dónde están los huecos |

**Votación en clase:** ¿qué opción elegirías, y por qué? Dos grupos que elijan distinto obtendrán clasificaciones distintas a partir de los mismos datos.

Elegimos la **A** (dejarlos como ausentes). No hay que hacer nada: pandas ya los ignora.

### 3.8 Resumir, pivotar y ordenar

**Agrupar y resumir** (*group by*, `groupby`): partimos la tabla por país, calculamos la media de cada parte y juntamos los resultados.

In [ ]:
means = period.groupby("country")["value"].mean().sort_values(ascending=False)
means.head(10)

### Tu turno 3.3

¿Cuál es el gasto medio de **España** (`"Spain"`) en el periodo, redondeado a 2 decimales? Guárdalo en `answer_3_3`.

<details><summary>Pista</summary>

`means` es una Series etiquetada por país: `means["Spain"]` selecciona un valor.
</details>

In [ ]:
answer_3_3 = None

In [ ]:
check("Ejercicio 3.3", answer_3_3 is not None and answer_3_3 == 4.28, 'means está etiquetada por país; selecciona Spain y después redondea')

¿Qué países gastan **menos**?

In [ ]:
means.dropna().tail(5)

**Pivotar** (*pivot*): convertir la lista larga (país, año, valor) en una **matriz** países × años. Las matrices son fáciles de leer y de ordenar.

In [ ]:
since_2018 = period[period["year"] >= 2018]
piv = since_2018.pivot_table(values="value", index="country", columns="year")
piv.head()

**Ordenar** (*rank*): ¿quién es el primero cada año? Solo ordenamos los países con **datos completos** para 2018–2023 (`dropna()` quita cualquier país con algún hueco).

In [ ]:
complete = piv.dropna()
print(len(complete), "países con datos completos para 2018-2023")
complete.rank(ascending=False, method="first").astype(int).head(8)

In [ ]:
totals = complete.sum(axis=1).sort_values(ascending=False)       # suma de los seis años
totals.head(5)

### Tu turno 3.4

¿Cuáles son los **3 primeros países** según `totals`? Guarda sus nombres como una lista en `answer_3_4`, del primero al tercero.

<details><summary>Pista</summary>

`totals.head(3)` da los 3 primeros. Sus nombres son el **índice**: `list(totals.head(3).index)`.
</details>

In [ ]:
answer_3_4 = None

In [ ]:
check("Ejercicio 3.4", answer_3_4 == ["Sweden", "Iceland", "Denmark"], "los nombres son el índice de totals.head(3)")

> La versión anterior de esta clase usaba la serie 1991–2011, y allí la primera era Dinamarca. Aquí es Suecia. Los países cambiaron, y también la definición: la serie se rehízo en 2012 con una nueva clasificación de los niveles educativos (ISCED 2011), así que las dos series no se pueden pegar sin más. **Misma pregunta, datos distintos, respuesta distinta.**

### 3.9 Comprobar el resultado

No te fíes nunca de un resultado que no has comprobado. Dos comprobaciones:

**(a) Contrastar con el otro archivo.** El CSV y el TSV deberían coincidir en las medias, **para el mismo periodo** (hay que comparar lo comparable). El CSV usa códigos (`DK`), el resultado del TSV usa nombres (`Denmark`):

In [ ]:
csv_period = edu[edu["TIME_PERIOD"].between(2014, 2023)]
csv_means = csv_period.groupby("geo")["OBS_VALUE"].mean()
print("Denmark  CSV:", round(csv_means["DK"], 2), "  TSV:", round(means["Denmark"], 2))
print("Spain    CSV:", round(csv_means["ES"], 2), "  TSV:", round(means["Spain"], 2))

**(b) Comprobar un valor a mano.** Elige un país y un año, búscalo en la tabla en bruto y búscalo en la limpia:

In [ ]:
print("en bruto:", raw.loc[raw["geo"] == "BE", "2020"].iloc[0])
tidy[(tidy["geo"] == "BE") & (tidy["year"] == 2020)][["country", "year", "value", "flag"]]

### Todos los pasos como funciones

Todo lo que hemos hecho paso a paso está empaquetado en el paquete auxiliar `etl` (carpeta `etl/`), para poder volver a ejecutarlo de una vez. Aquí tienes la función `tidy_tsv`: reconocerás cada línea.

In [ ]:
import inspect
from etl.transform import tidy_tsv, validate

print(inspect.getsource(tidy_tsv))

In [ ]:
packaged = tidy_tsv(raw, codes)
check("La función empaquetada da el mismo resultado",
      packaged.shape[0] == tidy.shape[0] and np.isclose(packaged["value"].sum(), tidy["value"].sum()),
      "algo difiere de los pasos anteriores")

### 3.10 Encontrar los errores sembrados

Los datos reales pueden fallar de más maneras que un valor ausente. En `data/exercises/educ_uoe_fine06_with_errors.tsv` el profesorado ha sembrado **cinco errores** en una copia del archivo. Tu trabajo: encontrarlos **con código**, como harías con datos que no puedes revisar a ojo.

Lo leemos con la función auxiliar `read_raw_tsv` (que también separa la primera columna, como en el paso 2.3) y lo limpiamos con la misma función:

In [ ]:
from etl.extract import read_raw_tsv

raw_bad = read_raw_tsv(EXERCISES / "educ_uoe_fine06_with_errors.tsv")
bad = tidy_tsv(raw_bad, codes, keep_raw=True)
print("forma:", bad.shape, "(el archivo limpio da (444, 7))")

### Tu turno 3.5

Responde estas cuatro preguntas sobre `bad`. Guarda cada número en la variable indicada.

1. `answer_dups`: ¿cuántas filas son **duplicados** de otra (mismo `geo` y `year`)?
2. `answer_range`: ¿cuántos valores están **por debajo de 0 o por encima de 15**? (El gasto es un % del PIB.)
3. `answer_unmatched`: ¿cuántas filas **no tienen nombre de país**?
4. `answer_unconverted`: ¿cuántas celdas **no se han podido convertir** en número? (El valor está ausente, pero el texto original **no** empezaba por `:`.)

<details><summary>Pistas</summary>

1. `bad.duplicated(["geo", "year"]).sum()`
2. `((bad["value"] < 0) | (bad["value"] > 15)).sum()`
3. `bad["country"].isnull().sum()`
4. `bad["value"].isnull() & ~bad["raw_cell"].str.strip().str.startswith(":")`, después `.sum()`
</details>

In [ ]:
answer_dups = None
answer_range = None
answer_unmatched = None
answer_unconverted = None

In [ ]:
check("Duplicados", answer_dups is not None and answer_dups == 12, 'duplicated([...]) y después sum()')
check("Fuera de rango", answer_range is not None and answer_range == 2, "cuenta los valores < 0 o > 15")
check("Sin nombre de país", answer_unmatched is not None and answer_unmatched == 12, 'isnull() sobre la columna country, después sum()')
check("No convertidos", answer_unconverted is not None and answer_unconverted == 1, "valor ausente + texto original que no empieza por ':'")

Ahora mira las **filas culpables** y averigua qué ha fallado en cada caso:

In [ ]:
print("Fuera de rango:")
display(bad[(bad["value"] < 0) | (bad["value"] > 15)][["geo", "year", "value", "raw_cell"]])

print("No se han podido convertir:")
display(bad[bad["value"].isnull() & ~bad["raw_cell"].str.strip().str.startswith(":")][["geo", "year", "value", "raw_cell"]])

print("Código desconocido:", sorted(bad.loc[bad["country"].isnull(), "geo"].unique()))
print("Código duplicado:", sorted(bad[bad.duplicated(["geo", "year"])]["geo"].unique()))

Estas comprobaciones también están empaquetadas en `validate(df)`. En un pipeline real las ejecutas **cada vez** y te paras si algo está mal:

In [ ]:
validate(bad)

### Punto de control: Transform

- ¿Cuál de los cinco errores se habría **tragado en silencio** un pipeline si no hubieras mirado? (Fíjate en el de *No convertidos*: el número se convirtió en un valor ausente sin ningún aviso.)
- ¿Por qué **"un valor ausente nuevo"** es una pista y no solo un hecho?
- Si fueran datos reales y no tuvieras las soluciones, ¿cómo sabrías que el resultado es correcto?

## Parte 4 · Load (cargar)

**Cargar** significa dejar el resultado **en algún sitio donde se pueda usar**. Nuestra tabla final: países de verdad, 2014–2023, con las columnas que necesitamos.

In [ ]:
final = period[["geo", "country", "year", "value", "flag"]].reset_index(drop=True)
print(final.shape)
final.head()

### 4.1 Guardarlo en tres formatos

| Formato | Sirve para |
|---|---|
| **CSV** | Universal y legible por cualquiera (Excel, R…). No recuerda los tipos de las columnas. |
| **Parquet** | Compacto y rápido; recuerda los tipos de las columnas. Bueno para datos grandes. |
| **SQLite** | Una *base de datos* en un solo archivo; puedes consultarla con SQL. El destino clásico de un ETL. |

In [ ]:
import sqlite3

PROCESSED.mkdir(exist_ok=True)
final.to_csv(PROCESSED / "education_spending.csv", index=False)
final.to_parquet(PROCESSED / "education_spending.parquet", index=False)

con = sqlite3.connect(PROCESSED / "education_spending.db")
final.to_sql("education_spending", con, if_exists="replace", index=False)

# Consultarla con SQL
query = """
    SELECT country, ROUND(AVG(value), 2) AS mean_spending
    FROM education_spending
    GROUP BY country
    ORDER BY mean_spending DESC
    LIMIT 5
"""
display(pd.read_sql(query, con))
con.close()

print("Guardados:", sorted(p.name for p in PROCESSED.glob("education_spending.*")))

### Tu turno 4.1

**Vuelve a leer el archivo Parquet** y cuenta sus filas. Guarda el número en `answer_4_1`. Un archivo guardado que nunca vuelves a leer es un archivo que no has comprobado.

<details><summary>Pista</summary>

`pd.read_parquet(path)` lo lee; `len(...)` cuenta las filas.
</details>

In [ ]:
answer_4_1 = None

In [ ]:
check("Ejercicio 4.1", answer_4_1 is not None and answer_4_1 == len(final), "pd.read_parquet(...) lo vuelve a leer; len(...) cuenta las filas")

### 4.2 Otros tipos de destino

SQLite es una **base de datos relacional**: tablas con filas y columnas, que se consultan con SQL; va bien para muchas lecturas y escrituras pequeñas (una aplicación, una web). No es el único tipo de lugar donde un ETL puede cargar su resultado. Aquí tienes dos más, con los mismos datos:

| Destino | Modelo | Va bien para | Ejemplos |
|---|---|---|---|
| **Base de datos relacional** | Tablas (filas) + SQL | Aplicaciones: muchas lecturas y escrituras pequeñas | SQLite, PostgreSQL, MySQL |
| **Base de datos analítica** | Tablas guardadas por columnas + SQL | Análisis: resúmenes sobre millones de filas | **DuckDB**, BigQuery, Snowflake |
| **Base de datos documental (NoSQL)** | Un documento JSON por objeto, sin columnas fijas | Datos con campos anidados o que cambian | MongoDB, CouchDB, Firestore |

#### Una base de datos analítica: DuckDB

**DuckDB** es "SQLite para el análisis": una base de datos en un solo archivo, sin servidor, que guarda los datos **por columnas** (como Parquet). Eso hace que resúmenes como medias o totales sean muy rápidos. Cargamos nuestra tabla y la consultamos con SQL:

In [ ]:
import duckdb   # https://duckdb.org/docs/stable/clients/python/overview

con = duckdb.connect(PROCESSED / "education_spending.duckdb")
con.register("final_df", final)                      # permite que SQL vea nuestro DataFrame con el nombre final_df
con.execute("CREATE OR REPLACE TABLE education_spending AS SELECT * FROM final_df")

display(con.sql("""
    SELECT country, ROUND(AVG(value), 2) AS mean_spending
    FROM education_spending
    GROUP BY country
    ORDER BY mean_spending DESC
    LIMIT 5
""").df())                                           # .df() convierte el resultado en un DataFrame de pandas
con.close()

DuckDB también puede ejecutar SQL **directamente sobre un archivo Parquet**, sin cargarlo antes en una base de datos. Así trabajan muchos equipos de datos: archivos Parquet en una carpeta (un *data lake*) y SQL por encima. La media de todos los países, año a año:

In [ ]:
parquet_file = (PROCESSED / "education_spending.parquet").as_posix()

duckdb.sql(f"""
    SELECT year, ROUND(AVG(value), 2) AS mean_spending, COUNT(value) AS countries_with_data
    FROM read_parquet('{parquet_file}')
    GROUP BY year
    ORDER BY year
""").df()

Fíjate en dos cosas: otra vez el salto de **2020**, y la columna `countries_with_data`, que cambia de un año a otro. Una media sobre 27 países y una media sobre 30 no son exactamente comparables.

#### Una base de datos documental (NoSQL)

Una **base de datos documental** no usa tablas. Guarda **documentos**: un objeto JSON por cosa (aquí, por país), que puede contener listas y otros objetos anidados. Dos documentos no tienen por qué tener los mismos campos. La más conocida es **MongoDB**, que guarda exactamente este tipo de JSON.

Construimos un documento por país con Python normal (una lista de diccionarios) y lo guardamos como archivo JSON con la biblioteca estándar:

In [ ]:
import json   # https://docs.python.org/3/library/json.html

documents = []
for (geo, country), group in final.groupby(["geo", "country"]):
    document = {
        "geo": geo,
        "country": country,
        # un valor por año; un valor ausente se convierte en None (null en JSON)
        "spending": {str(year): (None if pd.isna(value) else float(value))
                     for year, value in zip(group["year"], group["value"])},
    }
    # los flags solo donde los hay: los documentos pueden tener campos distintos
    flags = {str(year): flag for year, flag in zip(group["year"], group["flag"]) if isinstance(flag, str)}
    if flags:
        document["flags"] = flags
    documents.append(document)

with open(PROCESSED / "education_spending.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, indent=2, ensure_ascii=False)

print(len(documents), "documentos. El de España:")
spain = next(d for d in documents if d["country"] == "Spain")   # el primer documento cuyo país es Spain
print(json.dumps(spain, indent=2))

In [ ]:
sweden = next(d for d in documents if d["country"] == "Sweden")
print("Campos de España:", list(spain))
print("Campos de Suecia:", list(sweden), "  <- no tiene flags, así que no tiene el campo 'flags'")

Los mismos datos tienen ahora tres formas: **filas** (CSV, SQLite), **columnas** (Parquet, DuckDB) y **documentos** (JSON, MongoDB). Ninguna es "la correcta": depende de **quién los va a leer y cómo**.

### Tu turno 4.2

Usando `documents` (no `final`), encuentra el gasto de **España** (`"Spain"`) en **2020**. Guarda el número en `answer_4_2`.

<details><summary>Pista</summary>

El documento de España ya está en `spain`. Dentro, `spain["spending"]` es un diccionario cuyas claves son los años **como texto**: `spain["spending"]["2020"]`.
</details>

In [ ]:
answer_4_2 = None

In [ ]:
expected = final.loc[(final["country"] == "Spain") & (final["year"] == 2020), "value"].iloc[0]
check("Ejercicio 4.2", answer_4_2 is not None and answer_4_2 == expected,
      'spain["spending"] es un diccionario; la clave es el año como texto, "2020"')

### 4.3 Un primer vistazo a los datos: del ETL al EDA

El ETL está hecho: los datos están limpios, comprobados y guardados. El siguiente paso en cualquier proyecto de datos es el **EDA** (*Exploratory Data Analysis*, análisis exploratorio de datos): mirar los datos desde varios ángulos, con gráficos y resúmenes, para encontrar patrones, problemas y **preguntas**. El EDA será el próximo tema del curso; aquí solo echamos un primer vistazo.

Un gráfico es también la parte del resultado que más gente verá de verdad. Seguimos dos reglas: **ordenar las barras** y **usar pocos colores y con un propósito** (aquí, España está resaltada).

In [ ]:
TEAL, TERRACOTTA, INK, MUTED = "#0087A0", "#B4432B", "#16202E", "#5A626B"

bars = means.dropna().sort_values()               # orden ascendente, para que la barra más grande quede arriba
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(bars.index, bars.values, height=0.65,
        color=[TERRACOTTA if c == "Spain" else TEAL for c in bars.index])
for y, v in enumerate(bars.values):
    ax.text(v + 0.08, y, f"{v:.1f}", va="center", fontsize=9, color=INK)

ax.set_title("Gasto público medio en educación, 2014-2023", loc="left", fontsize=13, color=INK, pad=22)
ax.text(0, 1.01, "% del PIB. España resaltada. No se muestran los países sin datos.",
        transform=ax.transAxes, fontsize=9, color=MUTED)
ax.set_xlabel("% del PIB", color=MUTED)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False, colors=MUTED)
fig.tight_layout()
fig.savefig(OUTPUT / "mean_spending_by_country.png", dpi=150)
plt.show()

**Una pregunta para apuntar:** las barras son medias de 2014–2023, pero no todas las medias usan diez años. ¿Qué países se basan en muy pocos valores? El gráfico siguiente lo responde.

**¿Dónde están los huecos?** En Transform decidimos dejar los valores ausentes como ausentes (paso 3.7). Explorar los datos empieza por mirar **lo que no está**:

In [ ]:
from matplotlib.colors import ListedColormap   # https://matplotlib.org/stable/api/_as_gen/matplotlib.colors.ListedColormap.html

# Una fila por país, una columna por año; True donde falta el valor
missing_map = final.pivot(index="country", columns="year", values="value").isnull()
missing_map = missing_map.loc[missing_map.sum(axis=1).sort_values(kind="stable").index]   # primero los países con menos huecos

fig, ax = plt.subplots(figsize=(8, 9))
ax.imshow(missing_map.values, cmap=ListedColormap(["#CFE7EA", TERRACOTTA]), aspect="auto")
ax.set_yticks(range(len(missing_map.index)), missing_map.index, fontsize=9)
ax.set_xticks(range(len(missing_map.columns)), missing_map.columns, fontsize=9)
ax.set_title("¿Dónde están los valores ausentes?", loc="left", fontsize=13, color=INK, pad=22)
ax.text(0, 1.01, "Cada celda es un país en un año. Terracota = valor ausente.", transform=ax.transAxes, fontsize=9, color=MUTED)
ax.tick_params(length=0, colors=MUTED)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
plt.show()

**Preguntas para apuntar:** ¿los huecos son al azar o siguen un patrón (un país entero, los primeros años, los años después de 2019)? ¿Compararías Irlanda (`Ireland`), que solo tiene dos años, con los demás?

**¿Cómo de dispersos están los países?** Un *boxplot* (diagrama de caja) resume muchos valores en una figura: la caja contiene la mitad central de los países, la línea de dentro es la **mediana** y los puntos son **valores inusuales** (lejos del resto). Una caja por año:

In [ ]:
years = sorted(final["year"].unique())
by_year = [final.loc[final["year"] == year, "value"].dropna() for year in years]   # una lista de valores por año (sin los ausentes)

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(by_year, widths=0.55, patch_artist=True,
           boxprops=dict(facecolor="#CFE7EA", color=TEAL), medianprops=dict(color=TERRACOTTA, linewidth=2),
           whiskerprops=dict(color=TEAL), capprops=dict(color=TEAL), flierprops=dict(markeredgecolor=MUTED))
ax.set_xticks(range(1, len(years) + 1), years)
ax.set_title("¿Cómo de dispersos están los países, año a año?", loc="left", fontsize=13, color=INK, pad=22)
ax.text(0, 1.01, "% del PIB. Caja: la mitad central de los países; línea: la mediana; puntos: valores inusuales.", transform=ax.transAxes, fontsize=9, color=MUTED)
ax.set_ylabel("% del PIB", color=MUTED)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E2DED3", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

**Preguntas para apuntar:** toda la caja sube en **2020**: ¿es más gasto o menos PIB? ¿Qué países son los puntos de arriba? ¿Los países se acercan o se alejan entre sí con los años?

Ahora la **evolución en el tiempo**. Un gráfico de líneas necesita pocas líneas para seguir siendo legible: esta función acepta como máximo 4 países.

In [ ]:
def plot_countries(selected):
    """Gráfico de líneas del gasto a lo largo del tiempo para hasta 4 países."""
    assert len(selected) <= 4, "Más de 4 líneas cuesta de leer: elige como máximo 4 países"
    palette = ["#0087A0", "#B4432B", "#6B4FA0", "#A87A00"]
    series = period.pivot_table(values="value", index="year", columns="country")[selected]

    fig, ax = plt.subplots(figsize=(9, 5))
    for name, color in zip(selected, palette):
        ax.plot(series.index, series[name], color=color, linewidth=2, label=name)

    # Etiquetas directas al final de cada línea, separadas un poco para que no se solapen
    ends = sorted((series[n].dropna().iloc[-1], n) for n in selected)
    y_prev = -99
    for value, name in ends:
        y = max(value, y_prev + 0.25)
        ax.text(2023.2, y, name, va="center", fontsize=9, color=INK)
        y_prev = y

    ax.set_xlim(2014, 2025.2)
    ax.set_xticks(range(2014, 2024, 2))
    ax.margins(y=0.15)                     # espacio arriba para la leyenda
    ax.set_title("Gasto público en educación a lo largo del tiempo", loc="left", fontsize=13, color=INK)
    ax.set_ylabel("% del PIB", color=MUTED)
    ax.grid(axis="y", color="#E2DED3", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="upper left", ncol=len(selected))
    fig.tight_layout()
    plt.show()
    return fig


fig = plot_countries(["Spain", "Denmark", "Finland", "Germany"])
fig.savefig(OUTPUT / "spending_over_time.png", dpi=150)

### Tu turno 4.3

Cambia la lista de abajo por **cuatro países que tú elijas** y ejecuta la celda. (Para ver los nombres válidos: `sorted(period["country"].unique())`.) ¿Qué historias ves? ¿Alguna sorpresa en 2020? (Pista: el valor es el gasto **dividido por el PIB**. ¿Qué le pasó al PIB en 2020?)

In [ ]:
my_countries = ["Spain", "France", "Italy", "Portugal"]        # <-- cámbialos
_ = plot_countries(my_countries)

### Tus preguntas para el EDA

Una buena exploración produce **preguntas**, no solo gráficos. Escribe en la lista de abajo las preguntas que te han surgido con los gráficos (puedes dejar los ejemplos). Algunas no se pueden responder solo con estos datos: por ejemplo, *"¿los países que gastan más obtienen mejores resultados?"* necesita un segundo conjunto de datos. Para eso sirve el EDA, y será el próximo tema.

In [ ]:
my_questions = [
    "¿Por qué casi todos los países suben en 2020?",
    "¿Qué países tienen demasiados pocos años para compararlos?",
    # añade aquí tus propias preguntas, un texto por línea
]
for number, question in enumerate(my_questions, start=1):
    print(number, question)

### 4.4 Todo el ETL con una sola orden

El paquete auxiliar ejecuta **Extract → Transform → Load** desde cero, incluidas las comprobaciones de calidad. El mismo código, una sola línea:

In [ ]:
from etl.pipeline import run

result = run()
print("Idéntico a nuestro resultado paso a paso:", result.equals(final))

También puedes ejecutarlo desde un terminal con `uv run python -m etl.pipeline`. Si otra persona puede ejecutar tu pipeline y obtener el mismo resultado, el resultado es **fiable**: eso es la **reproducibilidad**.

### Punto de control: Load

- ¿Qué archivos han aparecido en `data/processed/` y en `output/`?
- Un compañero quiere abrir el resultado en Excel; una aplicación web necesita leer y actualizar un país cada vez; una analista quiere medias anuales de todo Eurostat. ¿Qué destino elegirías en cada caso, y por qué?
- ¿Podría un compañero o compañera repetir tu análisis usando solo esta carpeta del proyecto? ¿Qué necesitaría?
- Si Eurostat actualizara los datos mañana, ¿qué tendrías que cambiar? (Si el pipeline está bien hecho: nada.)

## Parte 5 · Reflexión

**1. Repasa tus predicciones** (el papel de la Parte 0).

- ¿Cuántos valores predijiste que faltaban? En el TSV, 99 celdas son `:` de 444, y 59 de ellas ni siquiera son filas en el CSV. Solo para nuestro periodo y los países reales, hay 65 valores ausentes de 350.
- ¿Qué país creías que gasta más? ¿Qué dicen los datos? (Suecia. Comprueba `means.head(1)`.)
- ¿El archivo se parecía a lo que imaginabas?

**2. Coméntalo con tu compañero/a**

- ¿Cuál fue la decisión más difícil en Transform? ¿Tu compañero/a y tú decidisteis lo mismo?
- ¿Qué paso no se te habría ocurrido sin el notebook?
- ¿De qué resultado no te fiarías sin comprobarlo? ¿Cómo lo comprobaste?

**3. Autoevaluación** (puntúa de 1 a 4)

1. Sé explicar qué es un DataFrame.
2. Sé explicar qué pasa en Extract, Transform y Load.
3. Sé decir qué decisiones tomé y por qué.
4. Sé cómo comprobar un resultado.
5. Sé qué haría si un conjunto de datos tuviera errores sembrados como el de hoy.

**4. Lo que viene: EDA**

Con el ETL hecho, los datos están listos para **explorarlos**. Ese es el próximo tema: el **EDA** (*Exploratory Data Analysis*, análisis exploratorio de datos), donde buscarás patrones, relaciones entre variables y valores inusuales y, al final, construirás un **cuadro de mando** que muestre los resultados a otras personas. Guarda tu lista `my_questions`: es un buen punto de partida.

**Para ir más allá**

- Repite los mismos pasos con otro conjunto de datos de Eurostat: `download_eurostat("<code>", DOWNLOADS)`. O descarga el `educ_uoe_fine06` completo con `key=""` y compara niveles educativos (`isced11`).
- Haz los ejercicios prácticos de la **Parte 6** (películas y valoraciones) con un conjunto de datos nuevo.
- Repite algunos pasos en polars (Apéndice A).
- Abre `etl/` y lee el código auxiliar. ¿Sabes cambiar el periodo en `run(first_year=..., last_year=...)`?

## Parte 6 · Manos a la obra: películas y valoraciones (práctica extra)

*Opcional: para casa, o para las parejas que terminen antes. Unos 45 minutos.*

Ahora vas a usar lo que has aprendido con un **conjunto de datos diferente**, con las preguntas de la hoja de ejercicios original. **MovieLens 1M** contiene cerca de **un millón de valoraciones** (de 1 a 5 estrellas) que 6.040 usuarios dieron a unas 3.900 películas.

Es la misma historia ETL:

| Fase | En esta parte |
|---|---|
| **Extract** | Leer tres archivos de texto con un separador poco habitual |
| **Transform** | Unir las tres tablas, filtrar, agrupar, pivotar, ordenar |
| **Load** | Responder preguntas con un resultado claro y comprobado |

### Cómo funciona cada ejercicio

Cada pregunta se divide en **pasos pequeños**. Para cada paso tienes:

1. **La idea** en palabras sencillas.
2. **Una receta**: un ejemplo diminuto con pocas filas que puedes leer de un vistazo.
3. **Tu turno**: sustituye `None` por tu código y ejecuta la celda.
4. **Dos pistas** si te atascas: la primera da la idea, la segunda es casi la respuesta.
5. **Una celda de comprobación** que dice `[OK]` o te da una pista.

**No** necesitas memorizar ninguna función. Copia el patrón de la receta y cambia los nombres.

### 6.0 Extract: consigue los tres archivos

El notebook busca los datos en `data/downloads/ml-1m/`. Si no están ahí, intenta descargarlos (unos 6 MB). Si no tienes internet, tu profesor/a te dará la carpeta: descomprímela en `data/downloads/` de forma que exista `data/downloads/ml-1m/ratings.dat`.

In [ ]:
from etl.movielens import find_movielens, download_movielens, read_movielens

ML_FOLDER = find_movielens(DOWNLOADS)
if ML_FOLDER is None:
    try:
        print("Descargando MovieLens 1M (unos 6 MB)...")
        ML_FOLDER = download_movielens(DOWNLOADS)
        print("Hecho.")
    except RuntimeError as err:
        print("La descarga ha fallado:", err)
        print("Pide a tu profesor/a la carpeta 'ml-1m' y ponla en data/downloads/")
else:
    print("Datos encontrados en", ML_FOLDER)


def safe(test):
    """Ejecuta una comprobación; si tu respuesta aún no está lista, devuelve False en lugar de mostrar un error."""
    try:
        return bool(test())
    except Exception:
        return False

Tres tablas, tres archivos. Fíjate en cómo se leen: es un buen ejemplo de **por qué Extract requiere cuidado**.

- Los campos están separados por **dos signos de dos puntos** `::`, no por una coma. Se lo indicamos a pandas con `sep="::"` (y `engine="python"`, que hace falta para un separador de dos caracteres).
- Los archivos **no tienen fila de cabecera**, así que damos nosotros los nombres de las columnas con `names=[...]` y `header=None`.
- Los títulos de las películas contienen acentos (é, ñ…), así que `movies.dat` se lee con `encoding="latin-1"`.

Todo eso ya está escrito en una función auxiliar.

In [ ]:
if ML_FOLDER is None:
    raise FileNotFoundError("No se han encontrado los datos de MovieLens: mira las instrucciones del paso 6.0")

users, ratings, movies = read_movielens(ML_FOLDER)

print("users  :", users.shape,   "(una fila por persona)")
print("ratings:", ratings.shape, "(una fila por valoración)")
print("movies :", movies.shape,  "(una fila por película)")
ratings.head()

In [ ]:
users.head()

In [ ]:
movies.head()

**Mira antes de continuar.** ¿Qué tabla dice *quién* valoró? ¿Cuál dice *qué* película? ¿Cuál tiene el **género** (`gender`)? ¿Cuál tiene el **título**?

La valoración sola (`user_id`, `movie_id`, `rating`) no nos dice el título ni el género. Para responder preguntas como *"¿qué películas gustan más a las mujeres que a los hombres?"* necesitamos **una tabla con todo**. Eso es una **unión** (*merge*, también llamada *join*).

### 6.1 Transform: une las tres tablas

**La idea.** Dos tablas comparten una columna (aquí `user_id`). `pd.merge(a, b)` las pone una al lado de la otra, emparejando las filas que tienen el mismo `user_id`. Por defecto, pandas encuentra por sí solo la columna compartida.

**La receta**, con tablas diminutas:

In [ ]:
mini_ratings = pd.DataFrame({"user_id": [1, 1, 2],
                             "movie_id": [10, 20, 10],
                             "rating": [5, 3, 4]})
mini_users   = pd.DataFrame({"user_id": [1, 2], "gender": ["F", "M"]})
mini_movies  = pd.DataFrame({"movie_id": [10, 20], "title": ["Alien", "Brazil"]})

step1 = pd.merge(mini_ratings, mini_users)     # añade el género, emparejando por user_id
step1

In [ ]:
pd.merge(step1, mini_movies)                   # añade el título, emparejando por movie_id

### Tu turno 6.1

Une las tablas reales: primero `ratings` con `users`, y luego el resultado con `movies`. Guarda el resultado en `data`.

<details><summary>Pista 1 (la idea)</summary>

Es la receta de arriba, pero con las tablas reales. Necesitas dos llamadas a `pd.merge`, una dentro de la otra, o una después de la otra.
</details>

<details><summary>Pista 2 (casi la respuesta)</summary>

`data = pd.merge(pd.merge(ratings, users), movies)`
</details>

In [ ]:
# Sustituye None por tu código
data = None

In [ ]:
check("Ejercicio 6.1",
      safe(lambda: len(data) == len(ratings)
           and {"title", "gender", "rating"} <= set(data.columns)
           and data["title"].notnull().all()),
      "después de la unión, `data` debe tener tantas filas como `ratings` y las columnas title, gender y rating")
data.head() if data is not None else None

Ahora cada fila de `data` es **una valoración**, con quién la dio (género, edad…) y para qué película (título, géneros cinematográficos).

### La caja de herramientas para las próximas preguntas

Cinco patrones bastan para toda la hoja. Estúdialos en esta tabla pequeña de 10 valoraciones (**lee la salida antes de seguir**).

In [ ]:
mini = pd.DataFrame({
    "user_id": [1, 1, 1, 2, 2, 3, 3, 3, 4, 4],
    "gender":  ["F", "F", "F", "M", "M", "F", "F", "F", "M", "M"],
    "title":   ["Alien", "Brazil", "Casablanca", "Alien", "Brazil",
                "Alien", "Brazil", "Casablanca", "Alien", "Dune"],
    "rating":  [5, 3, 4, 4, 1, 5, 4, 4, 3, 5],
})
mini

**Patrón A · cuenta las filas de cada grupo.**

In [ ]:
counts = mini.groupby("title").size()      # "para cada título, ¿cuántas filas?"
counts

**Patrón B · elige las etiquetas que cumplen una condición.** `counts >= 3` da True/False para cada título; al usarlo dentro de `counts.index[...]` se quedan los títulos que son True.

In [ ]:
frequent = counts.index[counts >= 3]       # títulos con al menos 3 valoraciones
list(frequent)

**Patrón C · quédate con las filas cuyo valor está en una lista** (`isin` significa "está en").

In [ ]:
mini_frequent = mini[mini["title"].isin(frequent)]
mini_frequent

**Patrón D · una tabla de medias, una fila por película y una columna por género** (`pivot_table`). Indicas qué columna contiene los **valores**, cuál da las **filas**, cuál da las **columnas** y cómo **combinar** los valores repetidos (aquí, la media).

In [ ]:
table = mini_frequent.pivot_table(values="rating", index="title", columns="gender", aggfunc="mean")
table

**Patrón E · ordena y quédate con las primeras filas.** `ascending=False` pone primero el más grande; `head(n)` se queda con los `n` primeros.

In [ ]:
table.sort_values(by="F", ascending=False).head(2)

Dos herramientas más, que verás más adelante: una **columna nueva a partir de un cálculo** (`table["diff"] = table["M"] - table["F"]`) y la **dispersión de las valoraciones** de una película (`groupby(...).std()`, la desviación estándar: pequeña cuando todo el mundo está de acuerdo, grande cuando las opiniones difieren).

---

### Pregunta 1 · ¿Qué películas tienen al menos 250 valoraciones?

**¿Por qué?** Una película valorada una sola vez por una persona con 5 estrellas tiene una media perfecta, pero esa media no significa nada. Nos quedamos solo con las películas con suficientes valoraciones. Es el mismo tipo de **decisión** que tomaste con los valores ausentes: apunta tu umbral.

**Receta:** Patrón A, luego B, luego C.

### Tu turno 6.2a · cuenta las valoraciones de cada película

Guarda en `ratings_per_title` el número de filas de `data` para cada `title`.

<details><summary>Pista 1</summary>

Patrón A: `groupby("title").size()`, pero sobre `data` en lugar de `mini`.
</details>

<details><summary>Pista 2</summary>

`ratings_per_title = data.groupby("title").size()`
</details>

In [ ]:
ratings_per_title = None

In [ ]:
check("Ejercicio 6.2a",
      safe(lambda: ratings_per_title.sum() == len(data) and ratings_per_title.index.is_unique),
      "necesitas un número por título, y todos los números juntos deben sumar el número de filas de `data`")

### Tu turno 6.2b · los títulos con al menos 250 valoraciones

Guarda en `active_titles` los títulos cuyo recuento es **250 o más**.

<details><summary>Pista 1</summary>

Patrón B. La condición es "mayor o igual que": `>=`.
</details>

<details><summary>Pista 2</summary>

`active_titles = ratings_per_title.index[ratings_per_title >= 250]`
</details>

In [ ]:
active_titles = None

In [ ]:
check("Ejercicio 6.2b",
      safe(lambda: len(active_titles) > 0
           and (ratings_per_title[active_titles] >= 250).all()
           and len(active_titles) == (ratings_per_title >= 250).sum()),
      "todos los títulos que te quedas necesitan 250 valoraciones o más, y no puede faltar ningún título con 250 o más")
print("Películas que se quedan:", len(active_titles) if active_titles is not None else "?")

### Tu turno 6.2c · quédate solo con las filas de valoraciones de esas películas

Guarda en `active` las filas de `data` cuyo `title` está en `active_titles`.

<details><summary>Pista 1</summary>

Patrón C: `data[data["title"].isin(...)]`.
</details>

<details><summary>Pista 2</summary>

`active = data[data["title"].isin(active_titles)]`
</details>

In [ ]:
active = None

In [ ]:
check("Ejercicio 6.2c",
      safe(lambda: set(active["title"]) == set(active_titles)
           and len(active) == ratings_per_title[active_titles].sum()),
      "`active` debe contener todas las valoraciones de las películas que te quedas, y nada más")
print("Filas que se quedan:", len(active) if active is not None else "?", "de", len(ratings))

**Piensa.** Han desaparecido muchas películas y se han quedado la mayoría de las valoraciones. ¿Por qué? (Pregúntale a tu compañero/a. Pista: ¿todas las películas son igual de populares?)

---

### Pregunta 2 · Valoración media de cada película, por género

**Receta:** Patrón D, sobre `active`.

### Tu turno 6.3

Guarda en `mean_ratings` una tabla con **una fila por título** y **una columna por género** (`F` y `M`), que contenga la valoración media.

<details><summary>Pista 1</summary>

Patrón D: values = `"rating"`, index = `"title"`, columns = `"gender"`, aggfunc = `"mean"`.
</details>

<details><summary>Pista 2</summary>

`mean_ratings = active.pivot_table(values="rating", index="title", columns="gender", aggfunc="mean")`
</details>

In [ ]:
mean_ratings = None

In [ ]:
def _expected_means():
    """La misma tabla, calculada de otra manera (agrupando y luego con unstack)."""
    return active.groupby(["title", "gender"])["rating"].mean().unstack()

check("Ejercicio 6.3",
      safe(lambda: set(mean_ratings.columns) == {"F", "M"}
           and set(mean_ratings.index) == set(active_titles)
           and np.allclose(mean_ratings[["F", "M"]].sort_index(), _expected_means()[["F", "M"]].sort_index())),
      "necesitas exactamente dos columnas, F y M, una fila por cada título que te quedas, con la valoración media")
mean_ratings.head() if mean_ratings is not None else None

---

### Pregunta 3 · ¿Qué películas valoran más alto las mujeres?

**Receta:** Patrón E.

### Tu turno 6.4

Guarda en `top_women` las **10 películas con la valoración media más alta entre las mujeres** (columna `F`).

<details><summary>Pista 1</summary>

Ordena `mean_ratings` por `"F"`, de mayor a menor, y quédate con las 10 primeras.
</details>

<details><summary>Pista 2</summary>

`top_women = mean_ratings.sort_values(by="F", ascending=False).head(10)`
</details>

In [ ]:
top_women = None

In [ ]:
check("Ejercicio 6.4",
      safe(lambda: len(top_women) == 10
           and top_women["F"].iloc[0] == mean_ratings["F"].max()
           and top_women["F"].is_monotonic_decreasing),
      "10 filas, ordenadas de la F más alta a la más baja")
top_women

---

### Pregunta 4 · ¿En qué no están de acuerdo hombres y mujeres?

Queremos tres listas: películas que gustan **más** a las mujeres que a los hombres, películas que gustan **más** a los hombres que a las mujeres, y las películas con las valoraciones más **discordantes** en general.

### Tu turno 6.5a · la diferencia entre hombres y mujeres

Guarda en `diff` la **media de los hombres menos la media de las mujeres** (`M` menos `F`) de cada película. La celda siguiente la añadirá a `mean_ratings` como una columna nueva llamada `diff`.

- Un `diff` **positivo**: los hombres la valoraron más alto.
- Un `diff` **negativo**: las mujeres la valoraron más alto.

<details><summary>Pista 1</summary>

Una columna de una tabla es una Series, y las Series se pueden restar: `table["M"] - table["F"]` hace el cálculo para todas las películas a la vez.
</details>

<details><summary>Pista 2</summary>

`diff = mean_ratings["M"] - mean_ratings["F"]`
</details>

In [ ]:
diff = None        # sustituye None por el cálculo

In [ ]:
check("Ejercicio 6.5a",
      safe(lambda: np.allclose(diff, mean_ratings["M"] - mean_ratings["F"])),
      "diff = M - F, en ese orden: el signo importa")

# Una columna nueva se crea asignándole valores
if diff is not None:
    mean_ratings["diff"] = diff

### Tu turno 6.5b · películas que gustan más a las mujeres que a los hombres

Guarda en `women_prefer` las **10 películas con el `diff` más negativo** (la mayor ventaja para las mujeres).

<details><summary>Pista 1</summary>

Patrón E, ordenando por `"diff"`. Preferidas por las mujeres = las más negativas = los valores **más pequeños** primero, así que `ascending=True`.
</details>

<details><summary>Pista 2</summary>

`women_prefer = mean_ratings.sort_values(by="diff", ascending=True).head(10)`
</details>

In [ ]:
women_prefer = None

In [ ]:
check("Ejercicio 6.5b",
      safe(lambda: len(women_prefer) == 10 and women_prefer["diff"].iloc[0] == mean_ratings["diff"].min()),
      "la primera fila debe tener el diff más pequeño (el más negativo)")
women_prefer[["F", "M", "diff"]] if women_prefer is not None else None

### Tu turno 6.5c · películas que gustan más a los hombres que a las mujeres

Guarda en `men_prefer` las **10 películas con el `diff` más grande**. Es lo mismo que en 6.5b, pero en la dirección contraria.

<details><summary>Pista 1</summary>

Cambia `ascending=True` por `ascending=False`.
</details>

<details><summary>Pista 2</summary>

`men_prefer = mean_ratings.sort_values(by="diff", ascending=False).head(10)`
</details>

In [ ]:
men_prefer = None

In [ ]:
check("Ejercicio 6.5c",
      safe(lambda: len(men_prefer) == 10 and men_prefer["diff"].iloc[0] == mean_ratings["diff"].max()),
      "la primera fila debe tener el diff más grande")
men_prefer[["F", "M", "diff"]] if men_prefer is not None else None

### Tu turno 6.5d · las películas más discordantes, sea cual sea el género

Una película es **discordante** si la gente no se pone de acuerdo sobre ella. Lo medimos con la **desviación estándar** de sus valoraciones (grande = las opiniones están dispersas).

1. Guarda en `rating_std` la desviación estándar de `rating` para cada `title`, usando solo `active` (las películas con suficientes valoraciones).
2. Guarda en `most_discordant` las 10 películas con la desviación estándar **más grande**.

<details><summary>Pista 1</summary>

Patrón A, pero con `["rating"].std()` en lugar de `.size()`, y después el Patrón E.
</details>

<details><summary>Pista 2</summary>

`rating_std = active.groupby("title")["rating"].std()` y
`most_discordant = rating_std.sort_values(ascending=False).head(10)`
</details>

In [ ]:
rating_std = None
most_discordant = None

In [ ]:
check("Ejercicio 6.5d",
      safe(lambda: set(rating_std.index) == set(active_titles)
           and len(most_discordant) == 10
           and most_discordant.iloc[0] == rating_std.max()),
      "una desviación estándar por cada título conservado; la primera de la lista es la más grande")
most_discordant

**Piensa.** ¿Parecen de *géneros distintos* las películas de `women_prefer` y de `men_prefer`? ¿Podría ser un **sesgo de los datos** (¿quién usa MovieLens?) más que una diferencia entre hombres y mujeres?

---

### Pregunta 5 · Valoración media de cada usuario, y la mejor película

### Tu turno 6.6a · la valoración media de cada usuario

Guarda en `mean_per_user` la media de `rating` que da cada `user_id`.

<details><summary>Pista 1</summary>

Patrón A con `user_id` como grupo y `["rating"].mean()`.
</details>

<details><summary>Pista 2</summary>

`mean_per_user = data.groupby("user_id")["rating"].mean()`
</details>

In [ ]:
mean_per_user = None

In [ ]:
check("Ejercicio 6.6a",
      safe(lambda: mean_per_user.index.is_unique
           and len(mean_per_user) == ratings["user_id"].nunique()
           and np.isclose(mean_per_user.loc[ratings["user_id"].iloc[0]],
                          ratings.loc[ratings["user_id"] == ratings["user_id"].iloc[0], "rating"].mean())),
      "un número por usuario; comprueba un usuario a mano")
print("El usuario más generoso da de media:", round(mean_per_user.max(), 2) if mean_per_user is not None else "?")

### Tu turno 6.6b · la película con la valoración media más alta (todas las películas)

Guarda en `best_film_all` el **título** de la película con la valoración media más alta, usando **todos** los datos. Consejo: `.idxmax()` devuelve la *etiqueta* del valor más grande.

<details><summary>Pista 1</summary>

Valoración media de cada título (como en 6.6a, pero agrupando por título), y después `.idxmax()`.
</details>

<details><summary>Pista 2</summary>

`best_film_all = data.groupby("title")["rating"].mean().idxmax()`
</details>

In [ ]:
best_film_all = None

In [ ]:
def _is_best(title, table):
    """¿Es `title` la película con la valoración media más alta de `table`?"""
    means = table.groupby("title")["rating"].mean()
    return means[title] == means.max()

check("Ejercicio 6.6b",
      safe(lambda: _is_best(best_film_all, data)),
      "la película cuya valoración media es igual a la valoración media máxima")
if best_film_all is not None:
    print(best_film_all, "->", (ratings["movie_id"].isin(movies.loc[movies["title"] == best_film_all, "movie_id"])).sum(), "valoraciones")

### Tu turno 6.6c · la mejor película, entre las películas con suficientes valoraciones

Haz lo mismo, pero solo con las películas que tienen 250 valoraciones o más (usa `active`). Guarda el título en `best_film_active`.

<details><summary>Pista 1</summary>

Igual que en 6.6b, sustituyendo `data` por `active`.
</details>

<details><summary>Pista 2</summary>

`best_film_active = active.groupby("title")["rating"].mean().idxmax()`
</details>

In [ ]:
best_film_active = None

In [ ]:
check("Ejercicio 6.6c",
      safe(lambda: _is_best(best_film_active, active)),
      "como en 6.6b, pero partiendo de `active`")
if best_film_active is not None:
    print(best_film_active, "->", (ratings["movie_id"].isin(movies.loc[movies["title"] == best_film_active, "movie_id"])).sum(), "valoraciones")

**Piensa.** ¿Son iguales las dos respuestas? Si no lo son, ¿cuál pondrías en un informe, y por qué? (Es el mismo tipo de decisión que «qué hacer con los valores ausentes»: mismos datos, regla distinta, conclusión distinta.)

---

### Pregunta 6 · Una función: las películas favoritas de un usuario

**¿Qué es una función?** Una receta con nombre. Le das una **entrada** y **devuelve** un resultado. La escribes una vez y la reutilizas con entradas distintas.

In [ ]:
def double(x):          # "double" es el nombre, "x" es la entrada
    return 2 * x        # "return" es el resultado

print(double(4), double(10))

Queremos `top_movies(user)`: las películas que este usuario valoró más alto. Vamos a construirla en dos pasos, primero para **un** usuario y después como función.

### Tu turno 6.7a · las valoraciones del usuario número 1

Guarda en `user_1` las filas de `data` en las que `user_id` vale `1`.

<details><summary>Pista 1</summary>

Filtra, como en la Parte 1: `data[data["user_id"] == 1]`. (Cuidado: dos signos de igual `==` para comparar.)
</details>

<details><summary>Pista 2</summary>

`user_1 = data[data["user_id"] == 1]`
</details>

In [ ]:
user_1 = None

In [ ]:
check("Ejercicio 6.7a",
      safe(lambda: len(user_1) == (ratings["user_id"] == 1).sum() and (user_1["user_id"] == 1).all()),
      "solo las filas del usuario 1, y todas ellas")

### Tu turno 6.7b · los 5 títulos que el usuario 1 valoró más alto

Guarda en `top5_user_1` los **títulos** de las 5 películas mejor valoradas de `user_1`. Ordena por `rating` de mayor a menor, quédate con 5 filas y después toma la columna `title`.

<details><summary>Pista 1</summary>

Patrón E sobre `user_1`, ordenando por `"rating"`; después selecciona la columna `"title"`.
</details>

<details><summary>Pista 2</summary>

`top5_user_1 = user_1.sort_values(by="rating", ascending=False).head(5)["title"]`
</details>

In [ ]:
top5_user_1 = None

In [ ]:
def _top_ok(user, result, n=5):
    """¿Son de verdad las n películas mejor valoradas por el usuario? (se admiten empates)"""
    rows = data[data["user_id"] == user]
    threshold = rows["rating"].nlargest(n).min()
    chosen = rows[rows["title"].isin(list(result))]
    return len(list(result)) == n and len(chosen) == n and (chosen["rating"] >= threshold).all()

check("Ejercicio 6.7b", safe(lambda: _top_ok(1, top5_user_1)),
      "5 títulos, cada uno valorado al menos tan alto como la quinta mejor valoración del usuario")

### Tu turno 6.7c · la función

Convierte los dos pasos en una función `top_movies(user, n=5)` que devuelva los `n` **títulos** mejor valorados de ese usuario. Sustituye el número de usuario `1` por la entrada `user`.

<details><summary>Pista 1</summary>

Copia tus dos líneas dentro de la función y cambia `1` por `user` y `5` por `n`. La última línea debe empezar por `return`.
</details>

<details><summary>Pista 2</summary>

```python
def top_movies(user, n=5):
    rows = data[data["user_id"] == user]
    return rows.sort_values(by="rating", ascending=False).head(n)["title"]
```
</details>

In [ ]:
def top_movies(user, n=5):
    # escribe aquí tu código, terminando con una línea que empiece por "return"
    return None

In [ ]:
_some_users = [int(u) for u in ratings["user_id"].drop_duplicates().iloc[[0, 1, -1]]]
check("Ejercicio 6.7c",
      safe(lambda: all(_top_ok(u, top_movies(u)) for u in _some_users)
           and _top_ok(_some_users[0], top_movies(_some_users[0], n=3), n=3)),
      "debe funcionar para cualquier usuario y cualquier n; pruébala con top_movies(2) y top_movies(2, n=3)")

In [ ]:
top_movies(2)

---

### Lo que has hecho, en términos de ETL

- **Extract:** has leído tres ficheros con un separador extraño, sin cabecera y con texto con acentos.
- **Transform:** has unido tres tablas en una, has filtrado con una *decisión* (al menos 250 valoraciones), has agrupado, has hecho pivot, has ordenado y has construido una función.
- **Load:** has obtenido respuestas y has **comprobado** cada una de una forma distinta a como la calculaste.

**Coméntalo con tu compañero/a**

1. ¿Qué ejercicio daría una respuesta distinta si cambiaras 250 por 50? Pruébalo.
2. En 6.6, ¿en qué respuesta confiarías? ¿Cómo justificarías tu elección ante un colega?
3. ¿Qué necesitarías para convertir este ejercicio en un pipeline reutilizable como el de la Parte 4?

## Apéndice A · Lo mismo en polars (opcional)

**polars** es una librería de DataFrames más reciente, normalmente más rápida y con un estilo más explícito. Las ideas son las mismas; cambia la sintaxis.

| Idea | pandas | polars |
|---|---|---|
| Leer un CSV | `pd.read_csv(path, usecols=[...])` | `pl.read_csv(path, columns=[...])` |
| Seleccionar una columna | `df["OBS_VALUE"]` | `df.select("OBS_VALUE")` |
| Filtrar filas | `df[df["OBS_VALUE"] > 6.5]` | `df.filter(pl.col("OBS_VALUE") > 6.5)` |
| Agrupar y calcular la media | `df.groupby("geo")["OBS_VALUE"].mean()` | `df.group_by("geo").agg(pl.col("OBS_VALUE").mean())` |
| Ordenar | `df.sort_values("x", ascending=False)` | `df.sort("x", descending=True)` |

In [ ]:
import polars as pl

edu_pl = pl.read_csv(RAW / "educ_uoe_fine06_long.csv", columns=["geo", "TIME_PERIOD", "OBS_VALUE"])

(edu_pl
   .filter(pl.col("TIME_PERIOD").is_between(2014, 2023))
   .group_by("geo")
   .agg(pl.col("OBS_VALUE").mean().alias("mean_spending"))
   .sort("mean_spending", descending=True, nulls_last=True)
   .head(5))

## Apéndice B · Glosario

| Término | Significado |
|---|---|
| **ETL** | Extract, Transform, Load (extraer, transformar, cargar): conseguir los datos, limpiarlos y dejarlos en un sitio donde sean útiles |
| **DataFrame** | Una tabla cuyas columnas tienen tipos y cuyas filas y columnas tienen etiquetas |
| **Series** | Una columna de un DataFrame |
| **Índice (index)** | Las etiquetas de las filas |
| **NaN** | «Not a number» (no es un número): un valor ausente. No es cero |
| **Flag** | Una nota de calidad asociada a un dato («estimado», «la definición difiere»…) |
| **Agregado (aggregate)** | Un grupo de países (EU27_2020) en lugar de un país |
| **Melt / unpivot** | Pasar de una tabla ancha (una columna por año) a una tabla larga (una fila por año) |
| **Pivot** | Pasar de una tabla larga a una matriz ancha |
| **Unión (join / merge)** | Emparejar dos tablas a través de una columna común |
| **Group by** | Dividir por categorías, calcular algo en cada parte y volver a juntarlo |
| **Parquet** | Un formato de fichero compacto, organizado por columnas, que conserva los tipos |
| **SQLite** | Una base de datos relacional guardada en un único fichero |
| **DuckDB** | Una base de datos analítica en un único fichero: guarda los datos por columnas y puede consultar archivos Parquet directamente |
| **Base de datos documental (NoSQL)** | Una base de datos que guarda documentos JSON en lugar de tablas (MongoDB) |
| **JSON** | Un formato de texto para datos anidados: objetos `{...}`, listas `[...]`, texto, números, `null` |
| **Tabla dinámica (pivot table)** | Una tabla de resúmenes (por ejemplo, medias) con una fila por categoría y una columna por categoría |
| **Desviación estándar (standard deviation)** | Lo dispersos que están los valores: pequeña si coinciden, grande si difieren |
| **Función (function)** | Una receta con nombre: recibe una entrada y devuelve un resultado |
| **Reproducibilidad (reproducibility)** | Cualquiera puede volver a ejecutar el proceso y obtener el mismo resultado |